# Input-Adaptive Hard-Exit DARTS (Colab-Ready)

This notebook is modular, reproducible, and designed for CIFAR-10 experiments with an input-adaptive architecture controller + hard early exit.

**Sections**
1. Setup
2. Config
3. Data
4. Search Ops
5. Model
6. Train/Eval Utilities
7. Train
8. Hard-Exit Sweep


In [ ]:
# 1) Setup
import os
import math
import time
import random
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as T

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Torch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('Device:', device)

amp_available = torch.cuda.is_available()
print('AMP enabled possible:', amp_available)


In [ ]:
# 2) Config
@dataclass
class CFG:
    data_root: str = './data'
    num_classes: int = 10
    batch_size: int = 128
    num_workers: int = 2

    # model
    C: int = 64
    num_nodes: int = 4

    # training
    epochs: int = 25
    lr_w: float = 3e-3
    lr_a: float = 3e-4
    weight_decay: float = 5e-4
    warmup_epochs: int = 5
    lambda_max: float = 0.05
    mu_reg: float = 1e-3
    grad_clip: float = 5.0
    amp: bool = True

    # inference thresholds
    tau_s: float = 0.45
    tau_c: float = 0.60
    conf_temp: float = 1.0

    # eval controls
    quick_eval_batches: int = 20
    sweep_eval_batches: int = 40
    calib_eval_batches: int = 40
    use_calib_candidates: bool = True

    # quick debug mode
    debug_quick: bool = False

cfg = CFG()

if cfg.debug_quick:
    cfg.epochs = 3
    cfg.C = 32
    cfg.batch_size = 64

print(cfg)

In [ ]:
# 3) Data
def build_dataloaders(cfg: CFG):
    train_tf = T.Compose([
        T.RandomCrop(32, padding=4),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    ])

    eval_tf = T.Compose([
        T.ToTensor(),
        T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    ])

    full_train = torchvision.datasets.CIFAR10(root=cfg.data_root, train=True, download=True, transform=train_tf)
    test_set = torchvision.datasets.CIFAR10(root=cfg.data_root, train=False, download=True, transform=eval_tf)

    val_size = 5000
    train_size = len(full_train) - val_size
    train_set, val_set = random_split(
        full_train,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42),
    )
    val_set.dataset.transform = eval_tf

    train_loader = DataLoader(train_set, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)
    test_loader = DataLoader(test_set, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = build_dataloaders(cfg)
print('Data ready:', len(train_loader), len(val_loader), len(test_loader))


In [ ]:
# 4) Search Ops
class Zero(nn.Module):
    def forward(self, x):
        return x.mul(0.0)

class Identity(nn.Module):
    def forward(self, x):
        return x

class SepConv3x3(nn.Module):
    def __init__(self, C):
        super().__init__()
        self.op = nn.Sequential(
            nn.ReLU(inplace=False),
            nn.Conv2d(C, C, 3, padding=1, groups=C, bias=False),
            nn.Conv2d(C, C, 1, bias=False),
            nn.BatchNorm2d(C),
            nn.ReLU(inplace=False),
            nn.Conv2d(C, C, 3, padding=1, groups=C, bias=False),
            nn.Conv2d(C, C, 1, bias=False),
            nn.BatchNorm2d(C),
        )
    def forward(self, x):
        return self.op(x)

class SepConv5x5(nn.Module):
    def __init__(self, C):
        super().__init__()
        self.op = nn.Sequential(
            nn.ReLU(inplace=False),
            nn.Conv2d(C, C, 5, padding=2, groups=C, bias=False),
            nn.Conv2d(C, C, 1, bias=False),
            nn.BatchNorm2d(C),
            nn.ReLU(inplace=False),
            nn.Conv2d(C, C, 5, padding=2, groups=C, bias=False),
            nn.Conv2d(C, C, 1, bias=False),
            nn.BatchNorm2d(C),
        )
    def forward(self, x):
        return self.op(x)

class AvgPool3x3(nn.Module):
    def __init__(self, C):
        super().__init__()
        self.op = nn.Sequential(
            nn.AvgPool2d(3, stride=1, padding=1, count_include_pad=False),
            nn.BatchNorm2d(C),
        )
    def forward(self, x):
        return self.op(x)

class MaxPool3x3(nn.Module):
    def __init__(self, C):
        super().__init__()
        self.op = nn.Sequential(
            nn.MaxPool2d(3, stride=1, padding=1),
            nn.BatchNorm2d(C),
        )
    def forward(self, x):
        return self.op(x)


In [ ]:
# 5) Model
class InputAdaptiveHardExitDARTS(nn.Module):
    def __init__(self, C=64, num_classes=10, num_nodes=4):
        super().__init__()
        self.C = C
        self.num_classes = num_classes
        self.num_nodes = num_nodes

        self.stem = nn.Sequential(
            nn.Conv2d(3, C, 3, padding=1, bias=False),
            nn.BatchNorm2d(C),
            nn.ReLU(inplace=True),
            nn.Conv2d(C, C, 3, padding=1, bias=False),
            nn.BatchNorm2d(C),
            nn.ReLU(inplace=True),
        )

        self.K = 6
        self.edge_meta = []
        for i in range(1, num_nodes + 1):
            for j in range(i):
                self.edge_meta.append((j, i))
        self.E = len(self.edge_meta)

        self.edge_ops = nn.ModuleList()
        for _ in range(self.E):
            self.edge_ops.append(nn.ModuleList([
                SepConv3x3(C),
                SepConv5x5(C),
                AvgPool3x3(C),
                MaxPool3x3(C),
                Identity(),
                Zero(),
            ]))

        self.alpha = nn.Parameter(1e-3 * torch.randn(self.E, self.K))

        self.arch_controller = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, 256), nn.ReLU(inplace=True),
            nn.Linear(256, self.E * self.K),
        )

        self.exit_heads = nn.ModuleList([nn.Linear(C, num_classes) for _ in range(num_nodes)])
        self.stop_heads = nn.ModuleList([nn.Linear(C, 1) for _ in range(num_nodes)])

        cost = torch.linspace(1.0 / num_nodes, 1.0, steps=num_nodes)
        self.register_buffer('exit_cost', cost)

    def _mixed_edge(self, x, w_edge, ops):
        out = 0.0
        for k, op in enumerate(ops):
            yk = op(x)
            wk = w_edge[:, k].view(-1, 1, 1, 1)
            out = out + wk * yk
        return out

    def forward_soft(self, x):
        B = x.size(0)
        stem = self.stem(x)

        delta = self.arch_controller(x).view(B, self.E, self.K)
        alpha_eff = self.alpha.unsqueeze(0) + delta
        p = F.softmax(alpha_eff, dim=-1)

        states = [stem]
        y_exits, s_list = [], []

        edge_idx = 0
        for node_i in range(1, self.num_nodes + 1):
            h_i = 0.0
            for from_j in range(node_i):
                w_edge = p[:, edge_idx, :]
                h_i = h_i + self._mixed_edge(states[from_j], w_edge, self.edge_ops[edge_idx])
                edge_idx += 1
            states.append(h_i)

            pooled = F.adaptive_avg_pool2d(h_i, 1).flatten(1)
            y_i = self.exit_heads[node_i - 1](pooled)
            beta_i = self.stop_heads[node_i - 1](pooled).squeeze(1)
            s_i = torch.sigmoid(beta_i)
            y_exits.append(y_i)
            s_list.append(s_i)

        y_exits = torch.stack(y_exits, dim=1)
        s = torch.stack(s_list, dim=1)

        q = []
        survival = torch.ones(B, device=x.device)
        for i in range(self.num_nodes):
            if i < self.num_nodes - 1:
                q_i = s[:, i] * survival
                q.append(q_i)
                survival = survival * (1.0 - s[:, i])
            else:
                q.append(survival)
        q = torch.stack(q, dim=1)

        y_hat = torch.einsum('bn,bnc->bc', q, y_exits)
        expected_cost = (q * self.exit_cost.view(1, -1)).sum(dim=1)
        delta_l2 = (delta ** 2).mean(dim=(1, 2))

        return {
            'y_hat': y_hat,
            'y_exits': y_exits,
            'q': q,
            's': s,
            'expected_cost': expected_cost,
            'delta_l2': delta_l2,
        }

    @torch.no_grad()
    def forward_hard_single(self, x1, tau_s=0.55, tau_c=0.75, conf_temp=1.0):
        stem = self.stem(x1)
        delta = self.arch_controller(x1).view(1, self.E, self.K)
        p = F.softmax(self.alpha.unsqueeze(0) + delta, dim=-1)

        states = [stem]
        edge_idx = 0
        logits = None

        conf_temp = max(float(conf_temp), 1e-3)

        for node_i in range(1, self.num_nodes + 1):
            h_i = 0.0
            for from_j in range(node_i):
                w_edge = p[:, edge_idx, :]
                h_i = h_i + self._mixed_edge(states[from_j], w_edge, self.edge_ops[edge_idx])
                edge_idx += 1
            states.append(h_i)

            pooled = F.adaptive_avg_pool2d(h_i, 1).flatten(1)
            logits = self.exit_heads[node_i - 1](pooled)
            s_i = torch.sigmoid(self.stop_heads[node_i - 1](pooled)).item()
            conf = F.softmax(logits / conf_temp, dim=-1).max(dim=-1).values.item()
            if ((s_i >= tau_s and conf >= tau_c) or (node_i == self.num_nodes)):
                return logits, node_i

        return logits, self.num_nodes

model = InputAdaptiveHardExitDARTS(C=cfg.C, num_classes=cfg.num_classes, num_nodes=cfg.num_nodes).to(device)
print('Model params:', sum(p.numel() for p in model.parameters()) / 1e6, 'M')

In [ ]:
# 6) Train / Eval Utilities
w_params = []
a_params = [model.alpha]
theta_params = list(model.arch_controller.parameters()) + list(model.stop_heads.parameters())
for n, p in model.named_parameters():
    if ('alpha' not in n) and ('arch_controller' not in n) and ('stop_heads' not in n):
        w_params.append(p)

opt_w = torch.optim.AdamW(w_params, lr=cfg.lr_w, weight_decay=cfg.weight_decay)
opt_a = torch.optim.AdamW(a_params, lr=cfg.lr_a, weight_decay=1e-4)
opt_t = torch.optim.AdamW(theta_params, lr=cfg.lr_a, weight_decay=1e-4)

sched_w = torch.optim.lr_scheduler.CosineAnnealingLR(opt_w, T_max=cfg.epochs)
sched_a = torch.optim.lr_scheduler.CosineAnnealingLR(opt_a, T_max=cfg.epochs)
sched_t = torch.optim.lr_scheduler.CosineAnnealingLR(opt_t, T_max=cfg.epochs)

amp_enabled = (cfg.amp and device.type == 'cuda')
scaler = torch.amp.GradScaler(device.type, enabled=amp_enabled)

def lambda_schedule(epoch: int):
    if epoch < cfg.warmup_epochs:
        return 0.0
    t = (epoch - cfg.warmup_epochs) / max(1, (cfg.epochs - cfg.warmup_epochs))
    return cfg.lambda_max * 0.5 * (1 - math.cos(math.pi * t))

@torch.no_grad()
def evaluate_soft(loader):
    model.eval()
    total_acc, total_cost, n = 0.0, 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model.forward_soft(x)
        total_acc += (out['y_hat'].argmax(1) == y).sum().item()
        total_cost += out['expected_cost'].sum().item()
        n += x.size(0)
    return {'acc': total_acc / n, 'avg_cost': total_cost / n}

@torch.no_grad()
def evaluate_hard(loader, tau_s=None, tau_c=None, conf_temp=None, max_batches=20):
    model.eval()
    tau_s = cfg.tau_s if tau_s is None else tau_s
    tau_c = cfg.tau_c if tau_c is None else tau_c
    conf_temp = cfg.conf_temp if conf_temp is None else conf_temp

    correct, n = 0, 0
    exits = []

    for bi, (x, y) in enumerate(loader):
        if (max_batches is not None) and (bi >= max_batches):
            break
        x, y = x.to(device), y.to(device)
        for i in range(x.size(0)):
            logits, exit_idx = model.forward_hard_single(
                x[i:i+1], tau_s=tau_s, tau_c=tau_c, conf_temp=conf_temp
            )
            pred = logits.argmax(dim=1)
            correct += (pred == y[i:i+1]).sum().item()
            exits.append(exit_idx)
            n += 1

    exits = np.array(exits) if len(exits) else np.array([])
    avg_exit = exits.mean() if len(exits) else float('nan')
    avg_cost = avg_exit / cfg.num_nodes if len(exits) else float('nan')
    return {
        'acc': correct / n if n else 0.0,
        'avg_exit_node': float(avg_exit),
        'avg_cost_norm': float(avg_cost),
        'exit_hist': {int(i): float((exits == i).mean()) for i in range(1, cfg.num_nodes + 1)} if len(exits) else {}
    }

@torch.no_grad()
def collect_calibration_stats(loader, conf_temp=None, max_batches=40):
    model.eval()
    conf_temp = cfg.conf_temp if conf_temp is None else conf_temp

    all_s = [[] for _ in range(cfg.num_nodes)]
    all_conf = [[] for _ in range(cfg.num_nodes)]
    all_correct = [[] for _ in range(cfg.num_nodes)]

    for bi, (x, y) in enumerate(loader):
        if (max_batches is not None) and (bi >= max_batches):
            break

        x, y = x.to(device), y.to(device)
        out = model.forward_soft(x)

        s = out['s']
        logits_nodes = out['y_exits']
        probs_nodes = F.softmax(logits_nodes / max(float(conf_temp), 1e-3), dim=-1)
        conf_nodes = probs_nodes.max(dim=-1).values
        pred_nodes = probs_nodes.argmax(dim=-1)
        correct_nodes = pred_nodes.eq(y.unsqueeze(1))

        for ni in range(cfg.num_nodes):
            all_s[ni].append(s[:, ni].detach().cpu())
            all_conf[ni].append(conf_nodes[:, ni].detach().cpu())
            all_correct[ni].append(correct_nodes[:, ni].float().detach().cpu())

    node_stats = []
    for ni in range(cfg.num_nodes):
        s_np = torch.cat(all_s[ni]).numpy() if all_s[ni] else np.array([])
        c_np = torch.cat(all_conf[ni]).numpy() if all_conf[ni] else np.array([])
        r_np = torch.cat(all_correct[ni]).numpy() if all_correct[ni] else np.array([])

        if len(s_np) == 0:
            node_stats.append({'node': ni + 1, 'n': 0})
            continue

        node_stats.append({
            'node': ni + 1,
            'n': int(len(s_np)),
            's_p25': float(np.percentile(s_np, 25)),
            's_p50': float(np.percentile(s_np, 50)),
            's_p75': float(np.percentile(s_np, 75)),
            'conf_p25': float(np.percentile(c_np, 25)),
            'conf_p50': float(np.percentile(c_np, 50)),
            'conf_p75': float(np.percentile(c_np, 75)),
            'acc_if_exit': float(r_np.mean()),
        })

    return {'conf_temp': float(conf_temp), 'nodes': node_stats}

def suggest_threshold_candidates(calib):
    s_vals, c_vals = [], []
    for d in calib['nodes']:
        if d.get('n', 0) <= 0:
            continue
        if d['node'] >= cfg.num_nodes:
            continue
        s_vals.extend([d['s_p25'], d['s_p50'], d['s_p75']])
        c_vals.extend([d['conf_p25'], d['conf_p50'], d['conf_p75']])

    if not s_vals:
        return [0.20, 0.30, 0.40, 0.50], [0.40, 0.50, 0.60, 0.70]

    s_vals = sorted(set(round(float(np.clip(v, 0.05, 0.95)), 2) for v in s_vals))
    c_vals = sorted(set(round(float(np.clip(v, 0.05, 0.95)), 2) for v in c_vals))

    if len(s_vals) > 6:
        idx = np.linspace(0, len(s_vals) - 1, 6).round().astype(int)
        s_vals = [s_vals[i] for i in idx]
    if len(c_vals) > 6:
        idx = np.linspace(0, len(c_vals) - 1, 6).round().astype(int)
        c_vals = [c_vals[i] for i in idx]

    return s_vals, c_vals

In [ ]:
# 7) Train
best_val = 0.0
best_state = None

for epoch in range(cfg.epochs):
    model.train()
    lam = lambda_schedule(epoch)

    running_loss, running_acc, running_cost, n_seen = 0.0, 0.0, 0.0, 0
    t0 = time.time()

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        opt_w.zero_grad(set_to_none=True)
        opt_a.zero_grad(set_to_none=True)
        opt_t.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
            out = model.forward_soft(x)
            ce = F.cross_entropy(out['y_hat'], y)
            cost = out['expected_cost'].mean()
            reg = out['delta_l2'].mean()
            loss = ce + lam * cost + cfg.mu_reg * reg

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(opt_w)
        scaler.step(opt_a)
        scaler.step(opt_t)
        scaler.update()

        bs = x.size(0)
        running_loss += loss.item() * bs
        running_acc += (out['y_hat'].argmax(1) == y).sum().item()
        running_cost += out['expected_cost'].sum().item()
        n_seen += bs

    sched_w.step()
    sched_a.step()
    sched_t.step()

    train_loss = running_loss / n_seen
    train_acc = running_acc / n_seen
    train_cost = running_cost / n_seen

    val_soft = evaluate_soft(val_loader)
    if val_soft['acc'] > best_val:
        best_val = val_soft['acc']
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    print(
        f"Epoch {epoch + 1:02d}/{cfg.epochs} | lam={lam:.4f} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} train_cost={train_cost:.4f} | "
        f"val_soft_acc={val_soft['acc']:.4f} val_soft_cost={val_soft['avg_cost']:.4f} | "
        f"time={time.time() - t0:.1f}s"
    )

if best_state is not None:
    model.load_state_dict(best_state)

print('Loaded best checkpoint by validation soft accuracy.')

In [ ]:
# 8) Final Eval + Hard-Exit Sweep
val_soft = evaluate_soft(val_loader)
test_soft = evaluate_soft(test_loader)
val_hard = evaluate_hard(
    val_loader,
    tau_s=cfg.tau_s,
    tau_c=cfg.tau_c,
    conf_temp=cfg.conf_temp,
    max_batches=cfg.quick_eval_batches,
)
test_hard = evaluate_hard(
    test_loader,
    tau_s=cfg.tau_s,
    tau_c=cfg.tau_c,
    conf_temp=cfg.conf_temp,
    max_batches=cfg.quick_eval_batches,
)

print('Val Soft:', val_soft)
print('Test Soft:', test_soft)
print('Val Hard Quick:', val_hard)
print('Test Hard Quick:', test_hard)

print('\n=== Calibration Summary (Validation) ===')
calib = collect_calibration_stats(
    val_loader,
    conf_temp=cfg.conf_temp,
    max_batches=cfg.calib_eval_batches,
)
for d in calib['nodes']:
    if d.get('n', 0) == 0:
        print(f"node={d['node']} | n=0")
        continue
    print(
        f"node={d['node']} | n={d['n']} | "
        f"s[p25,p50,p75]=({d['s_p25']:.3f},{d['s_p50']:.3f},{d['s_p75']:.3f}) | "
        f"conf[p25,p50,p75]=({d['conf_p25']:.3f},{d['conf_p50']:.3f},{d['conf_p75']:.3f}) | "
        f"acc_if_exit={d['acc_if_exit']:.3f}"
    )

default_taus_s = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60]
default_taus_c = [0.35, 0.45, 0.55, 0.65, 0.75, 0.85]

if cfg.use_calib_candidates:
    taus_s, taus_c = suggest_threshold_candidates(calib)
    print('\nUsing calibration-derived candidates:')
else:
    taus_s, taus_c = default_taus_s, default_taus_c
    print('\nUsing default sweep grid:')

print('taus_s:', taus_s)
print('taus_c:', taus_c)

conf_temps = [0.7, 1.0, 1.3]

best_pair = None
best_score = -1e9
for temp in conf_temps:
    print(f'--- conf_temp={temp:.2f} ---')
    for ts in taus_s:
        for tc in taus_c:
            m = evaluate_hard(
                val_loader,
                tau_s=ts,
                tau_c=tc,
                conf_temp=temp,
                max_batches=cfg.sweep_eval_batches,
            )
            score = m['acc'] - 0.08 * m['avg_cost_norm']
            print(
                f'tau_s={ts:.2f}, tau_c={tc:.2f} -> '
                f'acc={m["acc"]:.4f}, cost={m["avg_cost_norm"]:.4f}, exit={m["avg_exit_node"]:.2f}'
            )
            if score > best_score:
                best_score = score
                best_pair = (temp, ts, tc, m)

print('Best quick thresholds (temp, tau_s, tau_c, metrics):', best_pair)

if best_pair is not None:
    best_temp, best_ts, best_tc, _ = best_pair
    print('\n=== Test with best quick thresholds ===')
    test_best = evaluate_hard(
        test_loader,
        tau_s=best_ts,
        tau_c=best_tc,
        conf_temp=best_temp,
        max_batches=cfg.sweep_eval_batches,
    )
    print('Test Hard (best quick):', test_best)

os.makedirs('artifacts', exist_ok=True)
torch.save(
    {
        'model_state': model.state_dict(),
        'cfg': cfg.__dict__,
        'best_val_soft_acc': best_val,
        'best_hard_quick': best_pair,
        'calibration_val': calib,
    },
    'artifacts/input_adaptive_hard_exit_darts_ckpt.pt'
)
print('Saved checkpoint to artifacts/input_adaptive_hard_exit_darts_ckpt.pt')